# Emergence and stability of depth responses

Analysis of depth responses in neurons at day 1 to 5

In [ ]:
from v1_depth_map.revisions.revision_sessions import sessions
# Unfortunately the data for Day 1 of PZAH is missing (only 20 minutes of recording
# left, currently with synchronisation issue). Exclude the mouse

sphere_sessions = [(k,v) for k,v in sessions.items() if ('sphere' in v)]
mice = set([sess[0].split('_')[0] for sess in sphere_sessions])
print(f'{len(mice)} mice')
print(mice)

In [ ]:
# Load all neurons_df
import numpy as np
import flexiznam as flz
import pandas as pd

project = 'colasa_3d-vision_revisions'

neurons_dfs = dict()
for sess_name, protocol in sphere_sessions:
    mouse, sess = sess_name.split('_')
    neurons_df = pd.read_pickle(flz.get_processed_path(project) / mouse / sess / 'neurons_df.pickle')
    neurons_df['mouse'] = mouse
    neurons_df['session'] = sess_name
    neurons_df['uid'] = neurons_df.session.astype(str) + '_' + neurons_df.roi.astype(str)
    neurons_df['day'] = int(protocol.split('_')[1])
    neurons_dfs[sess_name] =neurons_df
neurons_df = pd.concat(neurons_dfs.values(), ignore_index=True)
print(f'Loaded {len(neurons_df)} neurons from {len(neurons_df.session.unique())} sessions')


In [ ]:
# Add depth tuned boolean
neurons_df["depth_tuned"] = neurons_df.apply(
    lambda x: x["depth_tuning_test_spearmanr_rval_closedloop"] > 0.1
    and x["depth_tuning_test_spearmanr_pval_closedloop"] < 0.05,
    axis=1,
)

In [ ]:
import matplotlib.pyplot as plt
from cottage_analysis.plotting import depth_selectivity_plots
fontsize_dict = {"title": 7, "label": 7, "tick": 5, "legend": 5}

ax = plt.subplot(1,1,1)
for mouse, mouse_df in neurons_df.groupby('mouse'):
    print(mouse)
    frac = np.zeros(5)
    for day, day_df in mouse_df.groupby('day'):
        valid = day_df[~np.isnan(day_df.depth_tuned)]
        frac[day-1] = (valid.depth_tuned.sum()/len(valid.depth_tuned))
        ax.text(day-1, frac[day-1]+0.01, len(valid.depth_tuned), fontsize=6, horizontalalignment='center', verticalalignment='bottom')
    ax.plot(np.arange(5), frac, 'o-', label=mouse)
ax.legend(loc='lower right')
ax.set_xticks(np.arange(5), labels=[f'Day {i}' for i in range(1,6)])
ax.set_ylabel('Proportion depth tuned')

In [ ]:
from pathlib import Path
# import roicat.util
import json
import warnings

root_path = '/Volumes/lab-znamenskiyp/home/shared/projects/colasa_3d-vision_revisions'
root_path = str(flz.get_processed_path(project))
neurons_df['tracking_id_manual'] = np.nan
neurons_df['tracking_uid_manual'] = ""


for mouse in mice:
    print(f'Loading data for {mouse}')
    roicat_path='/'.join([root_path, mouse, 'ROICat_spheretubes', 
        f"{mouse}_roicat.tracking.params_used.json"])

    with open(roicat_path) as roicat_file:
        roicat_params = json.load(roicat_file)
    session_list = []
    for statfile in roicat_params['data']['__init__']['paths_statFiles']:
        session_list.append(statfile.split(mouse)[1].split('/')[1])

    manual_click = Path('/'.join([root_path, mouse, 'ROICat_spheretubes', 
        'matches.json']))
    
    if manual_click.exists():
        print(f'Found manual click {manual_click}')
        # Load it
        with open(manual_click, 'r') as f:
            matches = json.load(f)
        matches = np.array(matches, dtype=float)
    else:
        print(f"No manual click for {mouse}")
        matches = None
        continue

    for sess_id, sess in enumerate(session_list):
        sess_name = f"{mouse}_{sess}"
        mask = (neurons_df.session==sess_name)

        if matches is None:
            continue
        # Add manual match
        manual_ids = np.arange(matches.shape[0])
        valid = ~np.isnan(matches[:, sess_id])

        valid_rois = matches[valid, sess_id].astype(int).astype(str)
        neuron_uids = sess_name + '_' + valid_rois
        valid_manual_ids = manual_ids[valid]
        # Create a dictionary mapping the neuron UID to the manual tracking ID
        uid_to_manual_id = dict(zip(neuron_uids, valid_manual_ids))
        
        # Update the tracking_id_manual column for the current session
        mask_sess = (neurons_df.session == sess_name)
        neurons_df.loc[mask_sess, 'tracking_id_manual'] = neurons_df.loc[mask_sess, 'uid'].map(uid_to_manual_id)


neurons_df['tracking_id_manual'] = np.nan_to_num(neurons_df['tracking_id_manual'], nan=-1)
mask = neurons_df.tracking_id_manual != -1 
neurons_df.loc[mask, 'tracking_uid_manual'] = (
    neurons_df.loc[mask, 'mouse'] + 
    neurons_df.loc[mask, 'tracking_id_manual'].astype(int).astype(str)
)

In [ ]:
tracked_neurons = neurons_df[neurons_df['tracking_uid_manual'] != ""]
tracking_ids = tracked_neurons.tracking_uid_manual.unique()

depth_tuning = np.zeros((len(tracking_ids), 5), float)+ np.nan
is_depth_tuned = np.zeros((len(tracking_ids), 5), float)+ np.nan
for itrack, tid in enumerate(tracking_ids):
    ndf = neurons_df[neurons_df.tracking_uid_manual==tid]
    for day, depth, is_tuned in zip(ndf.day, ndf.preferred_depth_closedloop, ndf.depth_tuned):
        depth_tuning[itrack, day-1] = depth
        is_depth_tuned[itrack, day-1] = is_tuned
    

In [ ]:
valid = np.all(np.nan_to_num(is_depth_tuned, nan=1)==1, axis=1)

d =depth_tuning[valid,:]
plt.imshow(np.log(d[np.argsort(d[:,0]),:]), aspect='auto', cmap='cool', interpolation='None')

In [ ]:
print(depth_tuning.shape)
logdepth_tuning = np.log(depth_tuning)
avg_d = np.nanmedian(logdepth_tuning, axis=1)
np.nanmin(logdepth_tuning), np.nanmax(logdepth_tuning)

In [ ]:
valid = np.all(np.nan_to_num(is_depth_tuned, nan=1)==1, axis=1)


tuning_change = logdepth_tuning - avg_d[:, None]
d =tuning_change[valid,:]
plt.imshow(d[np.argsort(depth_tuning[valid,:][:,0]),:], vmin=-2, vmax=2, cmap='RdBu', aspect='auto',interpolation='None')